In [5]:
import pandas as pd
import os
import numpy as np
import re
from google.colab import drive
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
# Důležitý nový import
import category_encoders as ce
import warnings
import matplotlib.pyplot as plt
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
#!pip install category_encoders

**Pokud chceme spustit výpočet pro jeden model, je třeba spustit funkce, které jsou označeny šipkou (udělal jsem to jako rychlý návod)**


# Loading data <-
Funkce pro načtení dat - je třeba spustit

In [7]:
def load_data(country):
  folder_path = f'/content/drive/MyDrive/ML2 semestralka/ML2-preprocessing/data/{country}/'
  try:
      league_folders = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]
  except FileNotFoundError:
      print(f"Chyba: Složka nebyla nalezena: {folder_path}")
      league_folders = []

  print(f"Nalezené složky lig: {league_folders}")

  all_dfs = []

  for league_name in league_folders:
      league_folder_path = os.path.join(folder_path, league_name)

      try:
          # Načtení souborů v dané lize
          files = [f for f in os.listdir(league_folder_path) if f.endswith('.csv')]

          # SEŘAZENÍ SOUBORŮ
          files.sort()

          if not files:
              print(f"Varování: Ve složce {league_folder_path} nebyly nalezeny žádné .csv soubory.")
              continue

          print(f" Načítám ligu: {league_name}, soubory: {files}")

          for csv_file in files:
              file_path = os.path.join(league_folder_path, csv_file)

              # Načtení dat
              df = pd.read_csv(file_path)
              df['League'] = league_name

              season_name = os.path.splitext(csv_file)[0]
              df['Season'] = season_name

              all_dfs.append(df)

      except Exception as e:
          print(f"Chyba při zpracování složky {league_folder_path}: {e}")

  if not all_dfs:
      print("Chyba: Nebyla načtena žádná data.")
      df_all = pd.DataFrame()
  else:
      df_all = pd.concat(all_dfs, ignore_index=True)
      print(f"\nÚspěšně načteno {len(df_all)} záznamů ze všech lig a sezón.")
      seasons = sorted(df_all['Season'].unique())

      if len(seasons) < 2:
          print("Chyba: K rozdělení potřebujete alespoň 2 sezóny dat.")
      else:
          last_season = seasons[-1]
          df_for_training = df_all[df_all['Season'] != last_season].copy()
          df_for_testing = df_all[df_all['Season'] == last_season].copy()

          print(f"Data rozdělena:")
          print(f" Poslední sezóna (test): {last_season} ({len(df_for_testing)} zápasů)")
          print(f" Ostatní sezóny (train): {seasons[:-1]} ({len(df_for_training)} zápasů)")
          return df_for_testing, df_for_training


In [8]:
#df_test, df_train = load_data('england')

# Feature engineering <-
Tady jsem přidal inkasované góly, tj. funkce __with_conceded, další pomocné funkce pro zkoušení výpočtu ale nebyly moc účinný (každopádně tam stejně jsou)..

In [9]:
def to_datetime(df):
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['IsWeekend'] = df['Date'].dt.dayofweek >= 5
    return df
def days_since_season_start(df):
    season_start = df['Date'].min()
    df['DaysFromSeasonStart'] = (df['Date'] - season_start).dt.days
    return df
def days_since_last_match(df):
    home = df[['Date', 'HomeTeam']].rename(columns={'HomeTeam': 'Team'})
    away = df[['Date', 'AwayTeam']].rename(columns={'AwayTeam': 'Team'})

    long_df = pd.concat([home, away]).sort_values(['Team', 'Date'])
    long_df['LastMatchDate'] = long_df.groupby('Team')['Date'].shift(1)
    homelastmatch= df.apply(
        lambda row: long_df.loc[(long_df['Team'] == row['HomeTeam']) & (long_df['Date'] < row['Date']), 'LastMatchDate'].max(),
        axis=1
    )

    awaylastmatch = df.apply(
        lambda row: long_df.loc[(long_df['Team'] == row['AwayTeam']) & (long_df['Date'] < row['Date']), 'LastMatchDate'].max(),
        axis=1
    )

    df['DaysSinceLastMatchAway'] = (df['Date'] - awaylastmatch).dt.days.fillna(365)
    df['DaysSinceLastMatchHome'] = (df['Date'] - homelastmatch).dt.days.fillna(365)
    df['DaysSinceLastMatchDiff'] = abs(df['DaysSinceLastMatchHome'] - df['DaysSinceLastMatchAway'])

    return df
def get_season_from_month(df):
    month_to_season = {
        1: 'Winter', 2: 'Winter', 12: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
    }

    df['four_seasons'] = df['Date'].dt.month.map(month_to_season)
    season_dummies = pd.get_dummies(df['four_seasons'], prefix='Season')
    df = pd.concat([df, season_dummies], axis=1).drop(columns=['four_seasons'])

    return df
def traintest_split(dfs, l):
    train_dfs = dfs[:-l]
    test_dfs = dfs[-l:]
    return train_dfs, test_dfs
def last_x_matches_agg(df, x, home_col='HomeTeam', away_col='AwayTeam'):
    df = df.sort_values(by='Date').reset_index(drop=True)

    home = df[['Date', home_col, 'FTHG']].rename(
        columns={home_col: 'Team', 'FTHG': 'Goals'}
    )
    away = df[['Date', away_col, 'FTAG']].rename(
        columns={away_col: 'Team', 'FTAG': 'Goals'}
    )

    long_df = pd.concat([home, away]).sort_values(['Team', 'Date'])

    long_df[f'AvgLast{x}_GoalsAgg'] = (
        long_df.groupby('Team')['Goals']
               .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
               .reset_index(level=0, drop=True)
    )

    df = df.merge(
        long_df[[ 'Team', 'Date', f'AvgLast{x}_GoalsAgg' ]],
        left_on=[home_col, 'Date'],
        right_on=['Team', 'Date'],
        how='left'
    ).rename(columns={f'AvgLast{x}_GoalsAgg': f'AvgLast{x}HomeGoalsAgg'}).drop(columns='Team')

    df = df.merge(
        long_df[[ 'Team', 'Date', f'AvgLast{x}_GoalsAgg' ]],
        left_on=[away_col, 'Date'],
        right_on=['Team', 'Date'],
        how='left'
    ).rename(columns={f'AvgLast{x}_GoalsAgg': f'AvgLast{x}AwayGoalsAgg'}).drop(columns='Team')

    return df


def compute_league_context(df):
    """
    Vypočítá ligové (Div) průměry – bez leakage (používá jen minulé zápasy).
    """

    # vytvoříme pomocné sloupce
    df = df.copy()
    df['TotalGoals'] = df['FTHG'] + df['FTAG']
    df['is_over25'] = (df['TotalGoals'] > 2).astype(int)
    df['TotalShots'] = df['HS'] + df['AS']
    df['TotalShotsOnTarget'] = df['HST'] + df['AST']

    # spočítáme kumulativní (expanding) průměry per liga
    df = df.sort_values(['Div', 'Date']).reset_index(drop=True)

    df['DivAvgGoals'] = (
        df.groupby('Div')['TotalGoals']
        .apply(lambda s: s.expanding().mean().shift(1))
        .reset_index(level=0, drop=True)
    )
    df['DivOver25Rate'] = (
        df.groupby('Div')['is_over25']
        .apply(lambda s: s.expanding().mean().shift(1))
        .reset_index(level=0, drop=True)
    )
    df['DivAvgShots'] = (
        df.groupby('Div')['TotalShots']
        .apply(lambda s: s.expanding().mean().shift(1))
        .reset_index(level=0, drop=True)
    )
    df['DivAvgShotsOnTarget'] = (
        df.groupby('Div')['TotalShotsOnTarget']
        .apply(lambda s: s.expanding().mean().shift(1))
        .reset_index(level=0, drop=True)
    )

    return df


def last_x_matches_agg_test(test_df,train_df,  x):
    test_df = test_df.copy()

    last_home_vals = (
        train_df[['HomeTeam', f'AvgLast{x}HomeGoalsAgg']]
        .dropna()
        .groupby('HomeTeam')[f'AvgLast{x}HomeGoalsAgg']
        .last()
    )

    last_away_vals = (
        train_df[['AwayTeam', f'AvgLast{x}AwayGoalsAgg']]
        .dropna()
        .groupby('AwayTeam')[f'AvgLast{x}AwayGoalsAgg']
        .last()
    )

    test_df[f'AvgLast{x}HomeGoalsAgg'] = test_df['HomeTeam'].map(last_home_vals)
    test_df[f'AvgLast{x}AwayGoalsAgg'] = test_df['AwayTeam'].map(last_away_vals)

    return test_df

def last_x_matches(df, x):
    df = df.sort_values(by='Date').reset_index(drop=True)

    df[f'AvgLast{x}HomeGoals'] = (
        df.groupby('HomeTeam')['FTHG']
          .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
          .reset_index(level=0, drop=True)
    )
    df[f'AvgLast{x}AwayGoals'] = (
        df.groupby('AwayTeam')['FTAG']
          .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
          .reset_index(level=0, drop=True)
    )

    return df

def last_x_matches_test(test_df, train_df, x):
    test_df = test_df.copy()

    last_home_vals = (
        train_df[['HomeTeam', f'AvgLast{x}HomeGoals']]
        .dropna()
        .groupby('HomeTeam')[f'AvgLast{x}HomeGoals']
        .last()
    )

    last_away_vals = (
        train_df[['AwayTeam', f'AvgLast{x}AwayGoals']]
        .dropna()
        .groupby('AwayTeam')[f'AvgLast{x}AwayGoals']
        .last()
    )

    test_df[f'AvgLast{x}HomeGoals'] = test_df['HomeTeam'].map(last_home_vals)
    test_df[f'AvgLast{x}AwayGoals'] = test_df['AwayTeam'].map(last_away_vals)

    return test_df

def merge_by_div(dfs, l):
    dfs = [df.copy() for df in dfs]

    merged_dfs = []

    for i in range(0, len(dfs), len(dfs)//l):
        group = dfs[i:i + len(dfs)//l]
        merged = pd.concat(group, ignore_index=True)
        merged_dfs.append(merged)

    return merged_dfs
def new_players(df, x):
    exp_mean_home = df['FTHG'].expanding(min_periods=1).mean().shift(1)
    df[f'AvgLast{x}HomeGoals'] = df[f'AvgLast{x}HomeGoals'].fillna(exp_mean_home)
    exp_mean_away = df['FTAG'].expanding(min_periods=1).mean().shift(1)
    df[f'AvgLast{x}AwayGoals'] = df[f'AvgLast{x}AwayGoals'].fillna(exp_mean_away)
    return df
def new_players_agg(df, x):
    exp_mean = ((df['FTHG'] + df['FTAG'])/2).expanding(min_periods=1).mean().shift(1)
    df[f'AvgLast{x}HomeGoalsAgg'] = df[f'AvgLast{x}HomeGoalsAgg'].fillna(exp_mean)
    df[f'AvgLast{x}AwayGoalsAgg'] = df[f'AvgLast{x}AwayGoalsAgg'].fillna(exp_mean)
    return df
def new_players_test_agg(test_df, train_df, x):
    div_means = (train_df.groupby('Div')['FTHG'].mean() + train_df.groupby('Div')['FTAG'].mean()) / 2

    test_df[f'AvgLast{x}HomeGoalsAgg'] = test_df[f'AvgLast{x}HomeGoalsAgg'].fillna(
    test_df['Div'].map(div_means)
    )
    test_df[f'AvgLast{x}AwayGoalsAgg'] = test_df[f'AvgLast{x}AwayGoalsAgg'].fillna(
    test_df['Div'].map(div_means)
    )
    return test_df
def new_players_test(test_df, train_df, x):
    div_means_home = train_df.groupby('Div')['FTHG'].mean()
    div_means_away = train_df.groupby('Div')['FTAG'].mean()

    test_df[f'AvgLast{x}HomeGoals'] = test_df[f'AvgLast{x}HomeGoals'].fillna(
    test_df['Div'].map(div_means_home)
    )
    test_df[f'AvgLast{x}AwayGoals'] = test_df[f'AvgLast{x}AwayGoals'].fillna(
    test_df['Div'].map(div_means_away)
    )
    return test_df
def encode_referee(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)

    df['TotalGoals'] = df['FTHG'] + df['FTAG']

    df['Referee_encoded'] = (
        df.groupby('Referee')['TotalGoals']
          .transform(lambda x: x.expanding().mean().shift(1))
    )

    df['Referee_encoded'] = df['Referee_encoded'].fillna(
        df['TotalGoals'].expanding().mean().shift(1)
    )

    df.drop(columns='TotalGoals', inplace=True)
    return df

def last_x_matches_total(df, x):
    df[f'AvgLast{x}Goals'] = df[f'AvgLast{x}HomeGoals'] + df[f'AvgLast{x}AwayGoals']
    return df
def last_x_matches_total_agg(df, x):
    df[f'AvgLast{x}GoalsAgg'] = df[f'AvgLast{x}HomeGoalsAgg'] + df[f'AvgLast{x}AwayGoalsAgg']
    return df

def add_betting_features(train_df, test_df):
    """Přepočet kurzů na pravděpodobnosti"""
    for df in [train_df, test_df]:

        for colset in [['B365H','B365D','B365A'], ['B365C>2.5','B365C<2.5']]:
            cols = [c for c in colset if c in df.columns]
            if len(cols) == 3:  # 1X2
                probs = 1 / df[cols]
                norm = probs.sum(axis=1)
                df[[f'p_{c}' for c in cols]] = probs.div(norm, axis=0)
            elif len(cols) == 2:  # over/under
                probs = 1 / df[cols]
                norm = probs.sum(axis=1)
                df[[f'p_{c}' for c in cols]] = probs.div(norm, axis=0)

        if 'B365C>2.5' in df.columns:
            df['logit_over25'] = np.log(df['B365C<2.5'] / df['B365C>2.5'])

    return train_df, test_df

def last_x_conceded(df, x):
    """
    Rolling průměry inkasovaných gólů za posledních x zápasů (bez leakage).
    - HomeConceded = FTAG, když je tým doma
    - AwayConceded = FTHG, když je tým venku
    """
    df = df.sort_values('Date').reset_index(drop=True)

    # domácí tým inkasované
    df[f'AvgLast{x}HomeConceded'] = (
        df.groupby('HomeTeam')['FTAG']
          .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
          .reset_index(level=0, drop=True)
    )

    # venkovní tým inkasované
    df[f'AvgLast{x}AwayConceded'] = (
        df.groupby('AwayTeam')['FTHG']
          .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
          .reset_index(level=0, drop=True)
    )

    return df
def last_x_conceded_test(test_df, train_df, x):
    """
    Naplní inkasované rollingy v testu tím, co je v trainu naposledy známé pro daný tým.
    Nevyužívá budoucnost → žádný leakage.
    """
    test_df = test_df.copy()

    last_home_conc = (
        train_df[['HomeTeam', f'AvgLast{x}HomeConceded']]
        .dropna()
        .groupby('HomeTeam')[f'AvgLast{x}HomeConceded']
        .last()
    )
    last_away_conc = (
        train_df[['AwayTeam', f'AvgLast{x}AwayConceded']]
        .dropna()
        .groupby('AwayTeam')[f'AvgLast{x}AwayConceded']
        .last()
    )

    test_df[f'AvgLast{x}HomeConceded'] = test_df['HomeTeam'].map(last_home_conc)
    test_df[f'AvgLast{x}AwayConceded'] = test_df['AwayTeam'].map(last_away_conc)

    return test_df
def new_players_conceded(df, x):
    """
    Vyplní NaN u train rollingů inkasovaných gólů „globální“ historií před aktuálním zápasem.
    Držíme se tvé strategie expanding().mean().shift(1).
    """
    # průměr inkasovaných doma (FTAG) přes dosavadní zápasy
    exp_mean_home_conc = df['FTAG'].expanding(min_periods=1).mean().shift(1)
    df[f'AvgLast{x}HomeConceded'] = df[f'AvgLast{x}HomeConceded'].fillna(exp_mean_home_conc)

    # průměr inkasovaných venku (FTHG) přes dosavadní zápasy
    exp_mean_away_conc = df['FTHG'].expanding(min_periods=1).mean().shift(1)
    df[f'AvgLast{x}AwayConceded'] = df[f'AvgLast{x}AwayConceded'].fillna(exp_mean_away_conc)

    return df

def new_players_test_conceded(test_df, train_df, x):
    """
    Vyplní NaN v testu ligovým průměrem inkasovaných gólů dle trainu (bez budoucnosti).
    """
    # průměr inkasovaných doma ~ průměr FTAG v lize; venku ~ průměr FTHG v lize
    div_mean_home_conc = train_df.groupby('Div')['FTAG'].mean()
    div_mean_away_conc = train_df.groupby('Div')['FTHG'].mean()

    test_df[f'AvgLast{x}HomeConceded'] = test_df[f'AvgLast{x}HomeConceded'].fillna(
        test_df['Div'].map(div_mean_home_conc)
    )
    test_df[f'AvgLast{x}AwayConceded'] = test_df[f'AvgLast{x}AwayConceded'].fillna(
        test_df['Div'].map(div_mean_away_conc)
    )
    return test_df

def fouls_corners_cards_offsides(df, x):
    """
    Rolling průměry týmových statistik za posledních X zápasů bez leakage.
    Chybějící hodnoty doplní týmovým expanding průměrem.
    """
    df = df.sort_values(by='Date').reset_index(drop=True)

    home_cols = ['HC', 'HF', 'HY', 'HST']
    away_cols = ['AC', 'AF', 'AY', 'AST']

    for col in home_cols:
        rolled = (
            df.groupby('HomeTeam')[col]
              .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
              .reset_index(level=0, drop=True)
        )
        # doplnění týmovým expanding mean
        filled = (
            df.groupby('HomeTeam')[col]
              .apply(lambda s: s.expanding(min_periods=1).mean().shift(1))
              .reset_index(level=0, drop=True)
        )
        df[f'AvgLast{x}{col}'] = rolled.fillna(filled)

    for col in away_cols:
        rolled = (
            df.groupby('AwayTeam')[col]
              .apply(lambda s: s.shift().rolling(window=x, min_periods=1).mean())
              .reset_index(level=0, drop=True)
        )
        filled = (
            df.groupby('AwayTeam')[col]
              .apply(lambda s: s.expanding(min_periods=1).mean().shift(1))
              .reset_index(level=0, drop=True)
        )
        df[f'AvgLast{x}{col}'] = rolled.fillna(filled)

    return df



def fouls_corners_cards_offsides_test(test_df, train_df, x):
    """
    Doplň rolling features do testu z posledních známých hodnot z tréninku.
    """
    test_df = test_df.copy()

    home_cols = ['HC', 'HF', 'HY', 'HST']
    away_cols = ['AC', 'AF', 'AY', 'AST']

    for col in home_cols:
        last_vals = (
            train_df[['HomeTeam', f'AvgLast{x}{col}']]
            .dropna()
            .groupby('HomeTeam')[f'AvgLast{x}{col}']
            .last()
        )
        test_df[f'AvgLast{x}{col}'] = test_df['HomeTeam'].map(last_vals)

    for col in away_cols:
        last_vals = (
            train_df[['AwayTeam', f'AvgLast{x}{col}']]
            .dropna()
            .groupby('AwayTeam')[f'AvgLast{x}{col}']
            .last()
        )
        test_df[f'AvgLast{x}{col}'] = test_df['AwayTeam'].map(last_vals)

    return test_df


# Komplet funkce pro tvorbu train/test vzorku. <-
Prakticky jsem to přepsal takto do jedné funkce, vzhledem k tomu, že tvorba setu trvá cca 1 minutu, tak jsem to neukládal.
 Jak jsem již avizoval, get_season_from_month nepoužívám -> funguje hůře než samotný month. Oboje je zbytečné mít v datasetu a month zachycuje i menší změny v samotném ročním období (na druhou stranu je citlivější na trendy). Zkusil jsem například udělat jednu feature "is_winter", kam jsem dal 12,1,2 měsíc a beztak to zhoršilo výsledek.
Is_weekend je třeba nepodceňovat, hodně mi to hýbalo s XGBoost (ostatním modelům jinak vyhovoval), ale ponechal jsem.
Proměnný jako počet branek, tyčky, žluté karty jsem testoval, všechny vedly ke všeobecnému zhoršení přesnosti.


In [10]:
def build_train_test(df_for_training, df_for_testing, l=1, x=5):
    """
    Kompletní pipeline pro vytvoření train/test sad s ošetřením všech featur.
    --------------------------------------------------
    dfs : list DataFrames (např. jednotlivé sezóny)
    l   : počet posledních DF určených pro test
    x   : rolling window (např. 5 posledních zápasů)
    --------------------------------------------------
    Vrací: train_df, test_df
    """

    train_df = df_for_training
    test_df = df_for_testing

    # Základní časové transformace
    train_df = to_datetime(train_df)
    test_df = to_datetime(test_df)
    train_df = get_season_from_month(train_df)
    test_df = get_season_from_month(test_df)
    train_df = days_since_season_start(train_df)
    test_df = days_since_season_start(test_df)
    train_df = days_since_last_match(train_df)
    test_df = days_since_last_match(test_df)

    # Jednoduché rolling agregáty
    train_df = last_x_matches(train_df, x)
    test_df = last_x_matches_test(test_df, train_df, x)
    train_df = last_x_matches_agg(train_df, x)
    test_df = last_x_matches_agg_test(test_df, train_df, x)

    # Rolling inkasované góly (train + test + fallbacky)
    train_df = last_x_conceded(train_df, x)
    test_df  = last_x_conceded_test(test_df, train_df, x)
    train_df = new_players_conceded(train_df, x)
    test_df  = new_players_test_conceded(test_df, train_df, x)

    # Ošetření týmů bez historie
    train_df = new_players(train_df, x)
    test_df = new_players_test(test_df, train_df, x)
    train_df = new_players_agg(train_df, x)
    test_df = new_players_test_agg(test_df, train_df, x)

    # Kódování rozhodčích
    try:
      train_df = encode_referee(train_df)
      test_df = encode_referee(test_df)
    except:
      print("Referee encoding failed")

    # Kombinace gólových ukazatelů
    train_df = last_x_matches_total(train_df, x)
    test_df = last_x_matches_total(test_df, x)
    train_df = last_x_matches_total_agg(train_df, x)
    test_df = last_x_matches_total_agg(test_df, x)

    #Branky, rohy, fouly, karty, shots_target..
    try:
      train_df = fouls_corners_cards_offsides(train_df, x)
      test_df = fouls_corners_cards_offsides_test(test_df,train_df, x)
    except:
      print("Fouls, corners, cards, offsides failed")

    # Kombinace gólových ukazatelů
    #train_df, test_df = add_betting_features(train_df, test_df)

    #for df in (train_df, test_df):
      #df['IsWinterWindow'] = df['Month'].isin([12,1,2]).astype(int)
    print("Building train/test set completed..")
    return train_df, test_df


In [11]:
def set_target_col(train_df, test_df):
  for df in [train_df, test_df]:
      df['TotalGoals'] = df['FTHG'] + df['FTAG']
      df['is_over25'] = (df['TotalGoals'] > 2).astype(int)
  return train_df, test_df

In [12]:
#missing = df.isnull().sum()                # spočítá prázdné hodnoty
#filtered = missing[missing < 10]          # nechá jen sloupce s méně než 100 NaN
#sorted_missing = filtered.sort_values()    # seřadí vzestupně podle počtu NaN
#print(sorted_missing)
#sorted_missing.to_csv("missing_values.csv")

# Použití sázek <-
Šance na over/out bude strong prediktor, zároveň je fajne vzít i šance na H/A/D, vybral jsem následující z Bets365 (mají jen 6 prázdných hodnot).
Nedával bych jich tam víc, budou jinak hodně korelovaný..

In [13]:
def bets_clear(train_df, test_df):
  # Seznam všech sloupců z bets, celý kód doplňuje NAN průměrem. Ano, tady se nepoužívá rolling
  # a jedná se v podstatě o data leakage, na druhou stranu jde jen o 6 hodnot, tak je to jedno
  all_bet_cols = ['B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5']

  for df in [train_df, test_df]:
      df.rename(columns={
          'B365>2.5': 'B365C_over',
          'B365<2.5': 'B365C_out',
          "Max>2.5": "Max_over",
          "Max<2.5": "Max_under"
      }, inplace=True)

  imputation_means = {}
  cols_to_fill = ['B365H', 'B365D', 'B365A', 'B365C_over', 'B365C_out']

  print("Počítám průměry z train_df...")
  for col in cols_to_fill:
      mean_val = train_df[col].mean()
      imputation_means[col] = mean_val
      print(f"Průměr pro {col}: {mean_val:.4f}")

  print("\nDoplňuji NaN v train_df a test_df...")
  for df in [train_df, test_df]:
      for col in cols_to_fill:
          df[col] = df[col].fillna(imputation_means[col])

  for df in [train_df, test_df]:
      numerator = df['B365C_out'].clip(1e-6)
      denominator = df['B365C_over'].clip(1e-6)
      df['logit_over25'] = np.log(numerator / denominator)

      df.replace([np.inf, -np.inf], np.nan, inplace=True)

  logit_mean = train_df['logit_over25'].mean()

  for df in [train_df, test_df]:
      df['logit_over25'] = df['logit_over25'].fillna(logit_mean)


  return train_df, test_df

In [14]:
def plot_bets(train_df, test_df):
  # vyber jen relevantní pravděpodobnosti
  prob_cols = ["Avg<2.5", "Avg>2.5", "Max_over", "Max_under", "P>2.5", "P<2.5", 'B365C_over', 'B365C_out']

  # spočítej korelace
  corrs = train_df[prob_cols + ['is_over25']].corr()['is_over25'].drop('is_over25').sort_values(ascending=False)

  # vykresli barplot
  plt.figure(figsize=(10, 5))
  corrs.plot(kind='bar', color='steelblue')
  plt.title('Korelace pravděpodobností bookmakerů s reálným výsledkem (is_over25)')
  plt.ylabel('Pearson correlation')
  plt.xticks(rotation=45, ha='right')
  plt.grid(True, axis='y', linestyle='--', alpha=0.7)
  plt.show()


  prob_cols = ["B365H", "B365D", "B365A", "BFA" , "BFD", "BFH", "PSD", "PSH", "PSA"]

  # spočítej korelace
  corrs = train_df[prob_cols + ['is_over25']].corr()['is_over25'].drop('is_over25').sort_values(ascending=False)

  # vykresli barplot
  plt.figure(figsize=(10, 5))
  corrs.plot(kind='bar', color='steelblue')
  plt.title('Korelace pravděpodobností bookmakerů s reálným výsledkem (is_over25)')
  plt.ylabel('Pearson correlation')
  plt.xticks(rotation=45, ha='right')
  plt.grid(True, axis='y', linestyle='--', alpha=0.7)
  plt.show()
  return None


# Imputace / čištění prázdných hodnot

In [15]:
# TROCHU LEAKAGE NO, ale pár hodnot nahradit průměrem neuškodí
def clean_data_mean(train_df, test_df):
  for col in train_df.columns:
    if train_df[col].isna().sum() > 0:
      train_df[col].fillna(train_df[col].mean(), inplace=True)
      test_df[col].fillna(test_df[col].mean(), inplace=True)
  return train_df, test_df


# Načtení dat pro training (v případě použití single modelu a single country) <-

In [16]:
df_data_test, df_data_train = load_data('england')
train_df, test_df = build_train_test(df_data_test, df_data_train, l=1, x=5)
train_df, test_df = set_target_col(train_df, test_df)
target_col = 'is_over25'
print(train_df['is_over25'].value_counts(normalize=True))
train_df, test_df = bets_clear(train_df, test_df)
#train_df, test_df = clean_data_mean(train_df, test_df) # Použít v případě jiného modelu než XGBoost
train_df.to_csv("train_processed.csv", index=False)
test_df.to_csv("test_processed.csv", index=False)


Nalezené složky lig: ['1', '0', '3', '2']
 Načítám ligu: 1, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']
 Načítám ligu: 0, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']
 Načítám ligu: 3, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']
 Načítám ligu: 2, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']

Úspěšně načteno 11952 záznamů ze všech lig a sezón.
Data rozdělena:
 Poslední sezóna (test): 2425 (2036 zápasů)
 Ostatní sezóny (train): ['1920', '2021', '2122', '2223', '2324'] (9916 zápasů)
Building train/test set completed..
is_over25
0    0.514244
1    0.485756
Name: proportion, dtype: float64
Počítám průměry z train_df...
Průměr pro B365H: 2.4678
Průměr pro B365D: 3.7041
Průměr pro B365A: 3.6909
Průměr pro B365C_over: 1.9299
Průměr pro B365C_out: 1.9612

Doplňuji NaN v train_df a test_df...


# 3 modely train
Basic trénink train/test pro ilustraci .
Použiju jednoduchej label_encoder pro týmy.
Některý týmy v hometeam nejsou v awayteam a naopak, takže nejdřív provedu concat, fitnu a provedu transform. Pokud je v testu něco, co není v train, tak dosadíme -1, ať se vyhneme leakage.
Rozjedem to rovnou na třech modelech

In [17]:
train_df.columns

Index(['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
       'HTHG', 'HTAG',
       ...
       'AvgLast5HF', 'AvgLast5HY', 'AvgLast5HST', 'AvgLast5AC', 'AvgLast5AF',
       'AvgLast5AY', 'AvgLast5AST', 'TotalGoals', 'is_over25', 'logit_over25'],
      dtype='object', length=166)

In [18]:
target_col = 'is_over25'

valid_features = [
    'DaysFromSeasonStart', 'DaysSinceLastMatchHome', 'DaysSinceLastMatchAway', 'DaysSinceLastMatchDiff',
    'Month', 'IsWeekend',
    'HomeTeam', 'AwayTeam',
    'AvgLast5HomeGoals', 'AvgLast5AwayGoals', 'AvgLast5Goals', # průměry z minulých 5 zápasů (hardcodovaný, pač jsem línej)
    'AvgLast5HomeGoalsAgg', 'AvgLast5AwayGoalsAgg',  'AvgLast5GoalsAgg', #agregovaný průměry
    'Referee_encoded',
    'AvgLast5HomeConceded', 'AvgLast5AwayConceded', # inkasovaný góly
    'logit_over25', 'B365H', 'B365D', 'B365A'
]

X_train = train_df[valid_features].fillna(0)
y_train = train_df[target_col]
X_test = test_df[valid_features].fillna(0)
y_test = test_df[target_col]



In [19]:
X_test.columns

Index(['DaysFromSeasonStart', 'DaysSinceLastMatchHome',
       'DaysSinceLastMatchAway', 'DaysSinceLastMatchDiff', 'Month',
       'IsWeekend', 'HomeTeam', 'AwayTeam', 'AvgLast5HomeGoals',
       'AvgLast5AwayGoals', 'AvgLast5Goals', 'AvgLast5HomeGoalsAgg',
       'AvgLast5AwayGoalsAgg', 'AvgLast5GoalsAgg', 'Referee_encoded',
       'AvgLast5HomeConceded', 'AvgLast5AwayConceded', 'logit_over25', 'B365H',
       'B365D', 'B365A'],
      dtype='object')

In [20]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

all_train_teams = pd.concat([X_train['HomeTeam'], X_train['AwayTeam']]).unique()
label_encoder.fit(all_train_teams)

X_train['HomeTeam'] = label_encoder.transform(X_train['HomeTeam'])
X_train['AwayTeam'] = label_encoder.transform(X_train['AwayTeam'])

# Týmy, které encoder nezná, ošetříme ručně:
def safe_transform(col):
    return col.apply(lambda x: label_encoder.transform([x])[0] if x in label_encoder.classes_ else -1)

X_test['HomeTeam'] = safe_transform(X_test['HomeTeam'])
X_test['AwayTeam'] = safe_transform(X_test['AwayTeam'])


In [21]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_log = logreg.predict(X_test)
y_proba_log = logreg.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_log))
print("ROC AUC:", roc_auc_score(y_test, y_proba_log))

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf))

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=50,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.56      0.43      0.49      5090
           1       0.52      0.64      0.57      4826

    accuracy                           0.53      9916
   macro avg       0.54      0.53      0.53      9916
weighted avg       0.54      0.53      0.53      9916

ROC AUC: 0.5602043042882487
=== Random Forest ===
              precision    recall  f1-score   support

           0       0.54      0.68      0.60      5090
           1       0.54      0.39      0.45      4826

    accuracy                           0.54      9916
   macro avg       0.54      0.53      0.53      9916
weighted avg       0.54      0.54      0.53      9916

ROC AUC: 0.5508952001152891
=== XGBoost ===
              precision    recall  f1-score   support

           0       0.53      0.69      0.60      5090
           1       0.53      0.37      0.43      4826

    accuracy                           0.53      9916
   mac

In [22]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC AUC': [
        roc_auc_score(y_test, y_proba_log),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb)
    ]
})

print("\n=== Model Performance Summary ===")
print(results.sort_values('ROC AUC', ascending=False).to_string(index=False))


=== Model Performance Summary ===
              Model  ROC AUC
Logistic Regression 0.560204
      Random Forest 0.550895
            XGBoost 0.541847


# XGBoost s target encodingem

In [23]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
# Důležitý nový import
import category_encoders as ce
import warnings

warnings.filterwarnings('ignore')

# --- 1. NAČTENÍ DAT ---
train_df = pd.read_csv("train_processed.csv")
test_df = pd.read_csv("test_processed.csv")
print("Používám data z paměti...")


# --- 2. DEFINICE FEATURES A CÍLE ---
# (Předpoklad: 'is_over25' je sloupec s 0/1)
target_col = 'is_over25'

# Rozdělíme features na numerické a kategoriální
NUMERIC_FEATURES = [
    'Month', 'IsWeekend',
    'Referee_encoded',
    'logit_over25', 'B365H', 'B365D', 'B365A',
    'DaysFromSeasonStart', 'DaysSinceLastMatchHome', 'DaysSinceLastMatchAway', 'DaysSinceLastMatchDiff',
    'AvgLast5GoalsAgg',
    'AvgLast5HomeGoals', 'AvgLast5AwayGoals',
    'AvgLast5HomeConceded', 'AvgLast5AwayConceded', # inkasovaný góly
]

CATEGORICAL_FEATURES = [
    'HomeTeam', 'AwayTeam'
]

valid_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# --- 3. PŘÍPRAVA X A Y ---
X_train = train_df[valid_features]
y_train = train_df[target_col]
X_test = test_df[valid_features]
y_test = test_df[target_col]

print("Data pro model připravena.")
print("Spouštím Target Encoding...")

# 'handle_unknown='mean'' automaticky vyřeší týmy v test_df,
# které nebyly v train_df (přiřadí jim průměr).
encoder = ce.TargetEncoder(cols=CATEGORICAL_FEATURES, handle_unknown='mean')

# Musíme 'fit' na X_train A y_train
encoder.fit(X_train, y_train)

# Až teď transformujeme oba sety
X_train_encoded = encoder.transform(X_train)
X_test_encoded = encoder.transform(X_test)

# Starý blok s LabelEncoderem už není potřeba!
print("Enkodování dokončeno.")

# --- 6. TRÉNOVÁNÍ MODELU ---
# Pro binární cíl (is_over25) použijeme XGBClassifier
print("Trénuji XGBClassifier...")
model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc', # Lepší metrika pro klasifikaci
    n_estimators=1000,
    learning_rate=0.01,
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42
)

model.fit(
    X_train_encoded, y_train,
    eval_set=[(X_test_encoded, y_test)],
    verbose=False
)
print("Trénování dokončeno.")

# --- 7. EVALUACE MODELU ---
print("\n--- EVALUACE MODELU (is_over25) ---")
preds = model.predict(X_test_encoded)
probs = model.predict_proba(X_test_encoded)[:, 1] # Pravděpodobnost pro 'Over'

accuracy = accuracy_score(y_test, preds)
roc_auc = roc_auc_score(y_test, probs)

print(f"Přesnost (Accuracy): {accuracy * 100:.2f}%")
print(f"ROC AUC: {roc_auc:.4f}")

Používám data z paměti...
Data pro model připravena.
Spouštím Target Encoding...
Enkodování dokončeno.
Trénuji XGBClassifier...
Trénování dokončeno.

--- EVALUACE MODELU (is_over25) ---
Přesnost (Accuracy): 51.40%
ROC AUC: 0.5222


# XGBoost hyperparameter train + test s best parameters

Train s rolling cross-validací. Předtím jsem v podstatě zkoušel učit model jen na train datech a pak validovat přes test - to nám řekne nejlepší výsledek na testu (poslední season), ale nepoví to, jak se chová model v real time prostředí (každé season zvlášť). Teď učíme model takhle:
Uděláme třeba 20 random kombinací parametrů (v podstatě 20 modelů) a každý z nich natrénuju na každém season (teda začnu na val=20, train=19 - potom pokračuju val=21, train=19+18 atd.) Nakonec vybereme model (parametry), který byl průměrně nejúspěšnější ve všech seasons. Poslední season se nepoužije, funguje jako test - model se na něm vyzkouší a to potom můžeme použít pro porovnání marže

In [24]:
target_col = 'is_over25'

valid_features = [
    'DaysFromSeasonStart', 'DaysSinceLastMatchHome', 'DaysSinceLastMatchAway', 'DaysSinceLastMatchDiff',
    'Month', 'Season', 'IsWeekend',
    'HomeTeam', 'AwayTeam',
    'AvgLast5HomeGoals', 'AvgLast5AwayGoals', # průměry z minulých 5 zápasů (hardcodovaný, pač jsem línej)
    'AvgLast5GoalsAgg', #agregovaný průměry
    'Referee_encoded',
    'AvgLast5HomeConceded', 'AvgLast5AwayConceded', # inkasovaný góly
    'logit_over25', 'B365D', 'B365A', 'B365H'
]

X_train = train_df[valid_features].fillna(0)
y_train = train_df[target_col]
X_test = test_df[valid_features].fillna(0)
y_test = test_df[target_col]
valid_features.remove("Season")



# Train/test na všech zemích s XGBoost + výpis features pro každou zemi

## Funkce pro zápis do csv všech features každého pro každý model

In [29]:
def xgb_importance_row(model, feature_names, importance_type="gain"):
    """
    Vrátí dict {feature_name: importance} pro jediný XGBoost model.
    Vstupy: model (XGB* nebo Booster), feature_names (sekvence/Index).
    """
    # 1) sklearn API (XGBRegressor/XGBClassifier)
    if hasattr(model, "feature_importances_"):
        s = pd.Series(model.feature_importances_, index=feature_names, dtype="float64")
        return s.to_dict()

    # 2) Booster API
    booster = getattr(model, "get_booster", lambda: None)()
    if booster is None:
        raise TypeError("Model nevypadá jako XGBoost (schází feature_importances_ i get_booster).")

    score = booster.get_score(importance_type=importance_type)  # např. {"f0": 0.1, ...} nebo {"featA": 0.1, ...}

    # mapování f0,f1,... -> názvy sloupců
    if all(k.startswith("f") and k[1:].isdigit() for k in score.keys()):
        score = {feature_names[int(k[1:])]: v for k, v in score.items()}

    # zarovnat na feature_names a chybějící vyplnit 0
    s = pd.Series(score, dtype="float64")
    s = s.reindex(feature_names).fillna(0.0)
    return s.to_dict()


## XGBoost model

In [27]:
def XGB_model_train(train_df, test_df, targer_col):

  # Rozdělíme features na numerické a kategoriální
  NUMERIC_FEATURES = [
      'Month', 'IsWeekend',
      'Referee_encoded',
      'logit_over25', 'B365H', 'B365D', 'B365A',
      'DaysFromSeasonStart', 'DaysSinceLastMatchHome', 'DaysSinceLastMatchAway', 'DaysSinceLastMatchDiff',
      'AvgLast5GoalsAgg',
      'AvgLast5HomeGoals', 'AvgLast5AwayGoals',
      'AvgLast5HomeConceded', 'AvgLast5AwayConceded', # inkasovaný góly
  ]
  for num_feature in NUMERIC_FEATURES:
    if num_feature not in train_df.columns:
      print(num_feature)
      NUMERIC_FEATURES.remove(num_feature)

  CATEGORICAL_FEATURES = [
      'HomeTeam', 'AwayTeam', 'League'
  ]

  valid_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

  # --- 3. PŘÍPRAVA X A Y ---
  X_train = train_df[valid_features]
  y_train = train_df[target_col]
  X_test = test_df[valid_features]
  y_test = test_df[target_col]

  print("Data pro model připravena.")
  print("Spouštím Target Encoding...")

  # 'handle_unknown='mean'' automaticky vyřeší týmy v test_df,
  # které nebyly v train_df (přiřadí jim průměr).
  encoder = ce.TargetEncoder(cols=CATEGORICAL_FEATURES, handle_unknown='mean')

  # Musíme 'fit' na X_train A y_train
  encoder.fit(X_train, y_train)

  # Až teď transformujeme oba sety
  X_train_encoded = encoder.transform(X_train)
  X_test_encoded = encoder.transform(X_test)

  # Starý blok s LabelEncoderem už není potřeba!
  print("Enkodování dokončeno.")

  # --- 6. TRÉNOVÁNÍ MODELU ---
  # Pro binární cíl (is_over25) použijeme XGBClassifier
  print("Trénuji XGBClassifier...")
  model = xgb.XGBClassifier(
      objective='binary:logistic',
      eval_metric='auc', # Lepší metrika pro klasifikaci
      n_estimators=1000,
      learning_rate=0.01,
      early_stopping_rounds=50,
      n_jobs=-1,
      random_state=42
  )

  model.fit(
      X_train_encoded, y_train,
      eval_set=[(X_test_encoded, y_test)],
      verbose=False
  )
  return model

## Feature importance pro každou zemi

In [30]:
countries = ["belgium", "england", "france", "germany", "greece", "italy", "netherlands", "portugal", "scotland", "spain", "turkey"]
fi_rows = []

for country in countries:

  df_data_test, df_data_train = load_data(country)
  train_df, test_df = build_train_test(df_data_test, df_data_train, l=1, x=5)
  train_df, test_df = set_target_col(train_df, test_df)
  target_col = 'is_over25'
  print(train_df['is_over25'].value_counts(normalize=True))
  train_df, test_df = bets_clear(train_df, test_df)
  model_to_use = XGB_model_train(train_df, test_df, target_col)
  row = xgb_importance_row(model_to_use, model_to_use.feature_names_in_)  # jen model + feature_names
  fi_rows.append(row)

fi_csv = pd.DataFrame(fi_rows)
fi_csv.to_csv("feature_importance.csv", index=False)

Nalezené složky lig: ['1']
 Načítám ligu: 1, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']

Úspěšně načteno 1774 záznamů ze všech lig a sezón.
Data rozdělena:
 Poslední sezóna (test): 2425 (312 zápasů)
 Ostatní sezóny (train): ['1920', '2021', '2122', '2223', '2324'] (1462 zápasů)
Referee encoding failed
Building train/test set completed..
is_over25
1    0.519231
0    0.480769
Name: proportion, dtype: float64
Počítám průměry z train_df...
Průměr pro B365H: 2.5006
Průměr pro B365D: 3.8175
Průměr pro B365A: 3.6647
Průměr pro B365C_over: 1.7513
Průměr pro B365C_out: 2.1178

Doplňuji NaN v train_df a test_df...
Referee_encoded
Data pro model připravena.
Spouštím Target Encoding...
Enkodování dokončeno.
Trénuji XGBClassifier...
Nalezené složky lig: ['1', '0', '3', '2']
 Načítám ligu: 1, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '2425.csv']
 Načítám ligu: 0, soubory: ['1920.csv', '2021.csv', '2122.csv', '2223.csv', '2324.csv', '